In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/FlyDrone-SNN
import os, sys
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
for p in ['flydrone/__init__.py', 'flydrone/train.py',
          'fly-brain/data/2025_Completeness_783.csv', 'fly-brain/data/2025_Connectivity_783.parquet']:
    assert os.path.exists(p), f'Eksik dosya: {p} (klasör yapısını kontrol et)'
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q optax pytest
!pip install -q pybullet git+https://github.com/utiasDSL/gym-pybullet-drones.git
!python -c "import jax; d = jax.devices(); print(jax.__version__, d); assert d[0].platform == 'gpu', 'GPU yok!'"

In [ ]:
!python -m pytest flydrone/tests -q -s
!python -m flydrone.scripts.grad_alignment --seeds 3 --out flydrone/results/grad_alignment.csv
!python -m flydrone.scripts.bench_full --batches 4 8 16 32

In [ ]:
import numpy as np
from flydrone.envs import DroneBatch
for act in ['one_d_rpm', 'rpm']:
    env = DroneBatch(n_envs=4, act=act)
    env.reset(); d = []
    for k in range(env.max_steps - 1):
        _, _, _, dist = env.step(env.teacher()); d.append(dist.mean())
    print(act, 'mean dist', np.mean(d), 'final', d[-1], 'crashes', env.pop_finished())
    env.close()

In [ ]:
## Add your training code and run